In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/Orginal.csv
/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


## Setup

In [2]:
import gc, random, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold        # KFold not StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEEDS   = [2026, 3407, 42, 123]   # 4 seeds → 4 submissions for voting
N_FOLDS = 5
TARGET  = 'Irrigation_Need'

## Load data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')
orig  = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/Orginal.csv')

train = pd.concat([train, orig], ignore_index=True)
print(f"Train: {train.shape}  Test: {test.shape}")

target2idx = {v: i for i, v in enumerate(train[TARGET].unique())}
idx2target = {v: k for k, v in target2idx.items()}
print(f"Label map: {target2idx}")

train[TARGET] = train[TARGET].map(target2idx)
y       = train[TARGET].values
test_id = test['id'].copy()

train.drop(columns=['id'], inplace=True)
test.drop(columns=['id'], inplace=True)

Train: (640000, 21)  Test: (270000, 20)
Label map: {'Low': 0, 'Medium': 1, 'High': 2}


## Feature Engineering 

In [4]:
CATS = [c for c in test.columns if test[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]
M    = train[NUMS].max()

def FE(df):
    out = df.copy()
    for c in NUMS:
        for k in range(-4, 4):
            out[f"{c}_digit{k}"] = (out[c] // (10**k) % 10).astype('int8')
        if M[c] < 10:
            out[c] = out[c].round(3)
        elif M[c] < 100:
            out[c] = out[c].round(2)
        else:
            out[c] = out[c].round(1)
    return out

train_fe = FE(train.drop(TARGET, axis=1))
test_fe  = FE(test)

# Drop constant cols
DROP = [c for c in test_fe.columns if test_fe[c].nunique() == 1]
print(f"Dropping {len(DROP)} constant cols")
train_fe.drop(columns=DROP, inplace=True)
test_fe.drop(columns=DROP, inplace=True)

# Encode categoricals (digit cols treated as categorical too)
CATEGORY = CATS + [c for c in test_fe.columns if 'digit' in c]
for c in CATEGORY:
    freq    = train_fe[c].value_counts()
    mapping = {v: i for i, (v, cnt) in enumerate(freq[freq >= 5].items())}
    defval  = len(mapping)
    train_fe[c] = train_fe[c].map(lambda x: mapping.get(x, defval))
    test_fe[c]  = test_fe[c].map(lambda x: mapping.get(x, defval))

FEATURES = CATEGORY + [c for c in NUMS if c in test_fe.columns]

Dropping 22 constant cols


## OrderedTE

In [5]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a
    def fit(self, train_df, category_cols, target_col):
        self.target_col = target_col
        self.category_cols = category_cols
        self.classes_ = sorted(train_df[target_col].unique())
        self.global_prior_ = train_df[target_col].value_counts(normalize=True).sort_index().values
        self._stats = {}
        for c in category_cols:
            stats = {}
            for k, cls in enumerate(self.classes_):
                y_bin = (train_df[target_col] == cls).astype(int)
                df    = train_df[[c]].copy()
                df['y']   = y_bin.values
                df['cnt'] = 1
                df['cum_cnt'] = df.groupby(c)['cnt'].cumsum() - df['cnt']
                df['cum_sum'] = df.groupby(c)['y'].cumsum() - df['y']
                pr    = self.a * self.global_prior_[k]
                te    = (df['cum_sum'] + pr) / (df['cum_cnt'] + self.a)
                te[df['cum_cnt'] == -1] = self.global_prior_[k]
                self.__dict__[f'_te_{c}_{cls}'] = te.values
                agg   = df.groupby(c)['y'].agg(['count','sum']).reset_index()
                agg.columns = [c, f'cnt_{cls}', f'sm_{cls}']
                agg[f'pr_{cls}'] = self.global_prior_[k]
                stats[cls] = agg
            self._stats[c] = stats
        return self
    def transform_train(self, train_df, category_cols):
        out = train_df.copy()
        for c in category_cols:
            for k, cls in enumerate(self.classes_):
                out[f'{c}_TE_cls{cls}'] = self.__dict__[f'_te_{c}_{cls}']
        return out
    def transform(self, df, category_cols):
        out = df.copy()
        for c in category_cols:
            for k, cls in enumerate(self.classes_):
                agg  = self._stats[c][cls]
                pr   = float(self.global_prior_[k])
                out  = out.merge(agg, on=c, how='left')
                te   = (out[f'sm_{cls}'].fillna(0) + self.a * pr) / (out[f'cnt_{cls}'].fillna(0) + self.a)
                out[f'{c}_TE_cls{cls}'] = te.fillna(pr).astype(np.float32)
                out.drop(columns=[f'cnt_{cls}',f'sm_{cls}',f'pr_{cls}'], inplace=True, errors='ignore')
        return out

## Sample weights

In [6]:
unique, counts = np.unique(y, return_counts=True)
weights_dict   = {cls: len(y) / len(unique) / cnt for cls, cnt in zip(unique, counts)}
sample_weights = np.array([weights_dict[lbl] for lbl in y])

## XGB params (Optuna-tuned)

In [7]:
XGB_PARAMS = dict(
    max_depth        = 3,
    learning_rate    = 0.021,
    n_estimators     = 3038,
    min_child_weight = 3,
    subsample        = 0.734,
    colsample_bytree = 0.505,
    colsample_bylevel= 0.704,
    colsample_bynode = 0.539,
    reg_alpha        = 5.7e-5,
    reg_lambda       = 7.843,
    gamma            = 0.018,
    objective        = 'multi:softprob',
    num_class        = 3,
    device           = 'cuda',
    tree_method      = 'hist',
    eval_metric      = 'mlogloss',
    n_jobs           = -1,
)

## Train 4 seeds

In [8]:
N = len(train_fe)
N_TEST = len(test_fe)

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"SEED={seed}")
    print(f"{'='*60}")
    np.random.seed(seed); random.seed(seed)

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)  # KFold not SKF

    oof_preds  = np.zeros((N, 3))
    test_preds = np.zeros((N_TEST, 3))

    for fold, (tri, vai) in enumerate(kf.split(train_fe)):
        Xtr = train_fe.iloc[tri].copy()
        Xva = train_fe.iloc[vai].copy()
        Xte = test_fe.copy()
        ytr = y[tri]
        yva = y[vai]
        wtr = sample_weights[tri]

        # OrderedTE — 4 shuffles, augmented training
        te = OrderedTE(a=1)
        tr_full = pd.concat([Xtr, pd.Series(ytr, name=TARGET, index=Xtr.index)], axis=1)
        tr_full['_w'] = wtr

        frames = []
        for i in range(4):
            sh = tr_full.sample(frac=1, random_state=seed+i).reset_index(drop=True)
            te.fit(sh, FEATURES, TARGET)
            enc = te.transform_train(sh, FEATURES)
            frames.append(enc)
        aug = pd.concat(frames, ignore_index=True)

        ytr_aug = aug[TARGET].values
        wtr_aug = aug['_w'].values
        aug.drop(columns=[TARGET, '_w'], inplace=True)
        aug.drop(columns=[c for c in CATEGORY if c in aug.columns], inplace=True)

        Xva_enc = te.transform(Xva, FEATURES)
        Xte_enc = te.transform(Xte, FEATURES)
        Xva_enc.drop(columns=[c for c in CATEGORY if c in Xva_enc.columns], inplace=True)
        Xte_enc.drop(columns=[c for c in CATEGORY if c in Xte_enc.columns], inplace=True)

        fc = [c for c in aug.columns if c in Xva_enc.columns]

        params = XGB_PARAMS.copy()
        params['random_state'] = seed

        m = XGBClassifier(**params)
        m.fit(aug[fc], ytr_aug, sample_weight=wtr_aug)

        oof_preds[vai]  = m.predict_proba(Xva_enc[fc])
        test_preds     += m.predict_proba(Xte_enc[fc]) / N_FOLDS

        s = balanced_accuracy_score(yva, np.argmax(oof_preds[vai], axis=1))
        print(f"  Fold {fold+1}/{N_FOLDS} bACC={s:.5f}")
        del m; gc.collect()

    cv = balanced_accuracy_score(y, np.argmax(oof_preds, axis=1))
    print(f"\nSEED={seed} OOF bACC = {cv:.5f}")

    # Optuna class weight optimization
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        cw = np.array([trial.suggest_float(f'cw{i}', 0.5, 3.0) for i in range(3)])
        adj = oof_preds * cw
        adj = adj / adj.sum(axis=1, keepdims=True)
        return balanced_accuracy_score(y, np.argmax(adj, axis=1))
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=200, show_progress_bar=False)
    best_cw = np.array([study.best_params[f'cw{i}'] for i in range(3)])
    print(f"Optuna bACC = {study.best_value:.5f}  cw={best_cw.round(3)}")

    # Apply class weights to test
    adj_test = test_preds * best_cw
    adj_test = adj_test / adj_test.sum(axis=1, keepdims=True)
    preds    = np.argmax(adj_test, axis=1)

    suf = ['a','b','c','d'][SEEDS.index(seed)]
    out = sub.copy()
    out['Irrigation_Need'] = [idx2target[p] for p in preds]
    out.to_csv(f'submission_seed{seed}_{suf}.csv', index=False)
    print(f"Saved submission_seed{seed}_{suf}.csv")

    # Also save raw argmax version for voting
    raw_preds = np.argmax(test_preds, axis=1)
    out_raw = sub.copy()
    out_raw['Irrigation_Need'] = [idx2target[p] for p in raw_preds]
    out_raw.to_csv(f'submission_seed{seed}_{suf}_raw.csv', index=False)

print("\nDONE — upload all 4 *_raw.csv files as a dataset for the voting notebook")


SEED=2026
  Fold 1/5 bACC=0.97988
  Fold 2/5 bACC=0.97996
  Fold 3/5 bACC=0.97997
  Fold 4/5 bACC=0.97936
  Fold 5/5 bACC=0.97971

SEED=2026 OOF bACC = 0.97978
Optuna bACC = 0.98011  cw=[1.096 1.009 1.438]
Saved submission_seed2026_a.csv

SEED=3407
  Fold 1/5 bACC=0.97999
  Fold 2/5 bACC=0.97928
  Fold 3/5 bACC=0.97993
  Fold 4/5 bACC=0.97851
  Fold 5/5 bACC=0.98061

SEED=3407 OOF bACC = 0.97966
Optuna bACC = 0.98007  cw=[1.548 1.749 2.371]
Saved submission_seed3407_b.csv

SEED=42
  Fold 1/5 bACC=0.97931
  Fold 2/5 bACC=0.97972
  Fold 3/5 bACC=0.98019
  Fold 4/5 bACC=0.97989
  Fold 5/5 bACC=0.97919

SEED=42 OOF bACC = 0.97965
Optuna bACC = 0.98005  cw=[1.791 2.061 2.985]
Saved submission_seed42_c.csv

SEED=123
  Fold 1/5 bACC=0.97976
  Fold 2/5 bACC=0.98047
  Fold 3/5 bACC=0.97978
  Fold 4/5 bACC=0.97833
  Fold 5/5 bACC=0.97864

SEED=123 OOF bACC = 0.97941
Optuna bACC = 0.97985  cw=[2.283 2.434 2.813]
Saved submission_seed123_d.csv

DONE — upload all 4 *_raw.csv files as a dataset for